# Pressão, parâmetros e o lugar dos hábitos

Usa o núcleo que passou no corte de repetibilidade e os indicadores que a troca de pressão move no entorno da mudança. A regressão controla mês e os outros parâmetros; o intervalo vem de bootstrap em blocos de 7 noites. Onde a regressão e as noites coladas à troca discordam, vale a comparação local: a pressão mudou junto com a época.

Rampa não entra quando não variou. O diário de hábitos está vazio de propósito; a grade N-de-1 mostra como um suplemento futuro seria testado.


In [ ]:
import numpy as np
import pandas as pd

from sleep_analysis.db import connect
from sleep_analysis.effects import apply_bh, estimate_effect, plan_n_of_1
from sleep_analysis.features import build_half_counts, build_night_features
from sleep_analysis.paths import PROJECT_DIR
from sleep_analysis.reliability import analysis_mask, load_indicator_config, score_indicators, select_core
from sleep_analysis.study import load_daily_log

REPORTS = PROJECT_DIR / "data" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
config = load_indicator_config()
connection = connect()
features = build_night_features(connection)
sample = features.loc[analysis_mask(features, config["usage_min_hours"]) & (features["period"] != "recovery")].copy()
scored = score_indicators(features, build_half_counts(connection), config)
components = scored["components"].set_index("indicator")
core = select_core(scored["components"], features, config)
outcomes = [
    "device_flow_lim_min_h", "rin", "snore_min_h", "hypopnea50_per_usage_h",
    "surge_index", "ahi", "stable_pct", "usage_hours",
]
print(f"{len(sample)} noites fora da recuperação, uso ≥ 4 h")
print("Passaram o corte:", ", ".join(core.loc[~core["below_icc"], "indicator"]) or "nenhum")
if sample["ramp_enabled"].nunique() < 2:
    print("Rampa não variou; não há efeito para estimar.")


In [ ]:
icc = components["icc"].to_dict()
rng = np.random.default_rng(7)
factors = {
    "pressure": ["month_index", "usage_hours", "epr_effective", "humidity_level"],
    "epr_effective": ["month_index", "usage_hours", "pressure", "humidity_level"],
    "humidity_level": ["month_index", "usage_hours", "pressure", "epr_effective"],
}
effects = []
for factor, covariates in factors.items():
    if sample[factor].nunique(dropna=True) < 2:
        continue
    effects.append(estimate_effect(
        sample, factor, outcomes, covariates=covariates, n_boot=400, block=7, icc_by_outcome=icc, rng=rng,
    ))
effects = apply_bh(pd.concat(effects, ignore_index=True))
effects.to_csv(REPORTS / "08_effects.csv", index=False)
local = components.loc[components.index.isin(outcomes), ["effect_per_cm", "n_switches", "responsiveness", "icc", "mdc95"]]
local.to_csv(REPORTS / "08_local_pressure.csv")
print("Efeito por cmH2O nas noites coladas a cada troca")
print(local.round(3).to_string())
print("\nRegressão no período inteiro, intervalo do bootstrap")
show = ["factor", "outcome", "n", "coefficient", "boot_low", "boot_high", "p_value", "q_value", "mde", "median_difference"]
print(effects[show].round(3).to_string(index=False))


In [ ]:
print("Leitura")
for factor, group in effects.groupby("factor", sort=False):
    print(f"\n{factor}")
    for row in group.itertuples():
        if row.note:
            print(f"  {row.outcome}: {row.note}")
            continue
        crosses_zero = row.boot_low < 0 < row.boot_high
        local_effect = components.at[row.outcome, "effect_per_cm"] if row.factor == "pressure" else np.nan
        status = "intervalo cruza zero" if crosses_zero else "intervalo fora do zero"
        local_text = ""
        if row.factor == "pressure" and pd.notna(local_effect):
            local_text = f"; nas trocas, {local_effect:.2f} por cmH2O"
            opposite = local_effect != 0 and not crosses_zero and (local_effect < 0) != (row.coefficient < 0)
            if opposite or (local_effect == 0 and not crosses_zero):
                local_text += " (discorda da regressão: vale a janela da troca)"
            elif local_effect != 0 and crosses_zero:
                local_text += " (a janela da troca aponta esse sentido; a regressão não confirma)"
        below_mde = pd.notna(row.mde) and abs(row.coefficient) < row.mde
        mde_text = " Abaixo do efeito mínimo de uma comparação de dois grupos." if below_mde else ""
        print(f"  {row.outcome}: {row.coefficient:.3f} ({status}, q {row.q_value:.3f}){local_text}.{mde_text}")


## Hábitos que ainda vão entrar

O catálogo está em `config/habits_catalog.json`. O diário `data/reference/daily_log.csv` tem uma linha por data e variável. `same_night` liga a data à noite ResMed desse dia; `previous_day` liga à noite do dia seguinte. Um suplemento se testa em blocos ABAB de 7 a 14 noites, não numa noite isolada.


In [ ]:
schedule = plan_n_of_1("2026-11-01", periods=4, block_nights=14)
schedule.to_csv(REPORTS / "08_n_of_1_example.csv", index=False)
log = load_daily_log()
print(f"Diário: {len(log)} registros. A grade abaixo é um exemplo, não um experimento iniciado.")
print(schedule.groupby("block")["phase"].first().to_string())
print(f"Primeira noite {schedule['night'].iloc[0]:%d/%m/%Y}, última {schedule['night'].iloc[-1]:%d/%m/%Y}.")
